# Evaluate baseline POWSM vs LoRA on Turkish chunks

**Colab:** Mount Drive, set paths in the config cell, run top-to-bottom.

**Local:** Paths auto-detect from repo — just run as-is.

Computes phone error rate using `edit_operations(pred, ref)` from `mod/assessment/`.
Also prints a per-phoneme confusion table for Turkish-specific phones.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q "espnet==202412" espnet-model-zoo "peft>=0.10" soundfile

In [ ]:
# ============================================================
# ⚙️  CONFIGURATION — edit these before running
# ============================================================

DRIVE_ROOT   = "/content/drive/MyDrive/senior"   # your Drive folder
CHUNKS_DIR   = "/content/turkish_chunks"          # local SSD copy
ADAPTER_DIR  = "/content/drive/MyDrive/senior/lora_checkpoints/best"

MODEL_ID     = "espnet/powsm"
LANG_SYM     = "<unk>"
TASK_SYM     = "<pr>"

# Set to an integer (e.g. 20) for a quick sanity check, None for the full test set
EVAL_SUBSET  = None

# ============================================================

In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    sys.path.insert(0, DRIVE_ROOT)              # turkish_lora_util.py
    sys.path.insert(0, f"{DRIVE_ROOT}/mod")    # assessment.edit_distance
else:
    _here = Path.cwd().resolve()
    FT = _here if (_here / "turkish_lora_util.py").is_file() else _here.parent
    REPO_ROOT = FT.parents[1]
    sys.path.insert(0, str(FT))
    sys.path.insert(0, str(REPO_ROOT / "mod"))
    CHUNKS_DIR  = str(FT / "data" / "turkish_chunks")
    ADAPTER_DIR = str(FT / "lora_checkpoints" / "best")

CHUNKS_DIR  = Path(CHUNKS_DIR)
ADAPTER_DIR = Path(ADAPTER_DIR)
print("CHUNKS_DIR :", CHUNKS_DIR,  "| exists:", CHUNKS_DIR.is_dir())
print("ADAPTER_DIR:", ADAPTER_DIR, "| exists:", ADAPTER_DIR.is_dir())

In [ ]:
# Copy chunks to local SSD on Colab (faster I/O during eval)
if IN_COLAB and not CHUNKS_DIR.is_dir():
    zip_path = f"{DRIVE_ROOT}/turkish_chunks.zip"
    print("Unzipping chunks to local SSD…")
    import subprocess
    subprocess.run(["unzip", "-q", zip_path, "-d", "/content/"], check=True)
    print("Done.", len(list(CHUNKS_DIR.glob("*.wav"))), "WAV files ready.")
else:
    print(len(list(CHUNKS_DIR.glob("*.wav"))), "WAV files on disk.")

In [ ]:
import json
import numpy as np
import soundfile as sf
import torch
from collections import defaultdict
from espnet2.bin.s2t_inference import Speech2Text
from assessment.edit_distance import edit_operations
from turkish_lora_util import patch_speech2text_lora

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

test_items = json.loads((CHUNKS_DIR / "test.json").read_text(encoding="utf-8"))
if EVAL_SUBSET:
    test_items = test_items[:EVAL_SUBSET]
print(f"Evaluating on {len(test_items)} test utterances")

In [ ]:
def parse_pr_tokens(raw: str) -> list:
    if "<notimestamps>" in raw:
        raw = raw.split("<notimestamps>", 1)[1]
    tokens = []
    for part in raw.strip().split("//"):
        part = part.strip().strip("/")
        if part:
            tokens.append(part)
    return tokens


def compute_per(pred: list, ref: list) -> float:
    ops = edit_operations(pred, ref)
    return len(ops) / max(len(ref), 1)


def run_pr(s2t_model: Speech2Text, wav_path: Path) -> list:
    speech, r = sf.read(wav_path)
    if r != 16000:
        raise ValueError(r)
    raw = s2t_model(np.asarray(speech, dtype=np.float32), text_prev="<na>")[0][0]
    return parse_pr_tokens(raw)


def eval_model(s2t_model: Speech2Text, items: list) -> tuple:
    """Returns (mean_per, per_phone_errors) where per_phone_errors is a confusion dict."""
    pers = []
    confusion = defaultdict(lambda: defaultdict(int))  # confusion[ref][pred] += 1
    for item in items:
        ref  = item["phones"]
        pred = run_pr(s2t_model, CHUNKS_DIR / f"{item['id']}.wav")
        pers.append(compute_per(pred, ref))
        # Build per-phone confusion from edit ops
        ops = edit_operations(pred, ref)
        for op in ops:
            if op[0] == "replace":
                confusion[op[2]][op[1]] += 1  # confusion[ref_phone][pred_phone]
            elif op[0] == "delete":
                confusion[op[1]]["<del>"] += 1
            elif op[0] == "insert":
                confusion["<ins>"][op[1]] += 1
    return float(np.mean(pers)), dict(confusion)

In [ ]:
print("Loading baseline POWSM…")
s2t_base = Speech2Text.from_pretrained(
    MODEL_ID, device=DEVICE, lang_sym=LANG_SYM, task_sym=TASK_SYM
)

print("Running baseline eval…")
base_per, base_confusion = eval_model(s2t_base, test_items)
print(f"Baseline mean PER: {base_per:.4f}  ({base_per*100:.1f}%)")

In [ ]:
if not ADAPTER_DIR.is_dir():
    print(f"⚠ Adapter not found at {ADAPTER_DIR} — skipping LoRA eval")
else:
    print("Loading LoRA model…")
    s2t_lora = Speech2Text.from_pretrained(
        MODEL_ID, device=DEVICE, lang_sym=LANG_SYM, task_sym=TASK_SYM
    )
    patch_speech2text_lora(s2t_lora, ADAPTER_DIR)

    print("Running LoRA eval…")
    lora_per, lora_confusion = eval_model(s2t_lora, test_items)
    print(f"LoRA    mean PER: {lora_per:.4f}  ({lora_per*100:.1f}%)")
    print(f"Delta           : {(lora_per - base_per)*100:+.1f} pp")

In [ ]:
# Per-phoneme analysis — focus on Turkish-specific phones
TUR_PHONES = ["ɯ", "œ", "ɟ", "ç", "ɰ", "ø", "y", "ɣ", "ʃ", "tʃ", "dʒ"]

if ADAPTER_DIR.is_dir():
    print(f"{'Phone':<8} {'Base errors':>12} {'LoRA errors':>12}")
    print("-" * 35)
    for ph in TUR_PHONES:
        base_errs = sum(base_confusion.get(ph, {}).values())
        lora_errs = sum(lora_confusion.get(ph, {}).values())
        marker = " ↓" if lora_errs < base_errs else (" ↑" if lora_errs > base_errs else "")
        print(f"/{ph}/<{'':>{7-len(ph)}} {base_errs:>12}  {lora_errs:>11}{marker}")